In [50]:
import torch

# Constants
kernel_size = 3
stride = 1
padding = 1

image_H = 224
image_L = 224
image_path = r"C:\Users\dwmcc\OneDrive\Documents\GitHub\SystolicArrayTransform\images\000320150002.jpg"
data_type = torch.float32  # Change to torch.int8 for int8 data type

in_channels = 3
out_channels = 72

In [52]:
from PIL import Image
import numpy as np
import os
from torchvision import transforms
from torch.nn import functional as F
import torch.nn as nn

# -------------------------------
# Custom im2col transformation
# -------------------------------
class PerformIm2Col:
    def __init__(self, kernel_size, stride, padding):
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def __call__(self, tensor):
        """
        Args:
            tensor (Tensor): Image tensor (C x H x W) with values in [0, 1]
        Returns:
            Tensor: resized to columar format (C * kernel_size * kernel_size, L)
        """
        if not torch.is_tensor(tensor):
            raise TypeError("Input should be a torch.Tensor")
        im2col_data = F.unfold(tensor, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding)
        return im2col_data

    def __repr__(self):
        return f"{self.__class__.__name__}(kernel_size={self.kernel_size}, stride={self.stride}, padding={self.padding})"

def get_image(image_path): 

    # Validate file existence
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")
    
    try:
        # Open image
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        raise ValueError(f"Error loading image: {e}")
    
    return image

# -------------------------------
# load and resize image function for matrix multiplication
# -------------------------------
def matmul_transform(image_data, kernel_size=3, stride=1, padding=1, data_type=torch.int8):
    """
    Loads an image from the given path and resizes it to the specified size.
    
    Args:
        image_data (pixel array): raw RBG input image data
        size (tuple): Desired output size (width, height).
        kernel_size (int): Size of the convolutional kernel.
        stride (int): Stride of the convolution.
        padding (int): Padding for the convolution.
        data_type (torch.dtype): Data type for the output tensor.
    
    Returns:
        torch.Tensor: Transformed image tensor.
    """
    
    # Define transformation: resize + convert to tensor + normalize + im2col + convert dataype
    transform = transforms.Compose([
        PerformIm2Col(kernel_size, stride, padding),  # Apply im2col transformation
        transforms.ConvertImageDtype(data_type), # Convert to target data type after im2col transformation
    ])
    
    return transform(image_data)


# -------------------------------
# load and resize image function for 2DConv
# -------------------------------
def nn_transform(image_data, size=(224, 224)):
    '''
    Transform image to specified size and normalize.

    Args:
        image_data (pixel array): raw RBG input image data
        size (tuple): Desired output size (width, height).
        data_type (torch.dtype): Data type for the output tensor.

        Returns:
            torch.Tensor: Transformed image tensor.
    '''

    transform = transforms.Compose([
            transforms.Resize(size),  # Resize to 224x224
            transforms.ToTensor(),    # Convert to tensor (C, H, W) in [0,1]
            transforms.Normalize(     # Normalize with ImageNet mean/std
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
            ])
        
    return transform(image_data)

def reconstruct_image(image_data, size=(224, 224), kernel_size=3, stride=1, padding=1):
    '''
    Reconstructs the original image from the im2col data using fold operation.

    Args:
        image_data (Tensor): The im2col data tensor.
        size (tuple): The original image size (width, height).
        kernel_size (int): Size of the convolutional kernel used in im2col.
        stride (int): Stride used in im2col.
        padding (int): Padding used in im2col.
    Returns:
        Tensor: Reconstructed image tensor.
    '''

    # Perform the folding operation
    folded = F.fold(image_data, output_size=size, kernel_size=kernel_size, stride=stride, padding=padding)
    
    return folded

In [5]:
raw_image = get_image(image_path)
image_data = nn_transform(raw_image, size=(image_H, image_L), data_type=data_type)
print(image_data.shape)  # Should print (1, 3, 224, 224)
image_data_matmul = matmul_transform(image_data, kernel_size=kernel_size, stride=stride, padding=padding, data_type=data_type)
print(image_data_matmul.shape)  # Should print (244, 244, 3)

torch.Size([3, 224, 224])
torch.Size([27, 50176])


In [6]:
# Weights initialized to image size with 3 channels (assuming RGB) and kernel size
weights = torch.ones((out_channels, in_channels, kernel_size, kernel_size), dtype=data_type)
weights_reshaped = weights.view(weights.size(0), -1)

Method 1: Matrix Multiply

In [7]:
def test_matmult(image_data, weights):
    # Apply matrix multiplication (convolution) using the im2col data and weights
    conv_output_matmult = torch.matmul(weights, image_data)
    # Compute H_out and W_out correctly
    H_out = ((image_H + 2 * padding - kernel_size) // stride) + 1
    W_out = ((image_L + 2 * padding - kernel_size) // stride) + 1

    # Reshape matmul_output directly (no need for F.fold)
    matmul_output = conv_output_matmult.view(out_channels, H_out, W_out)
    return matmul_output

matmul_output = test_matmult(image_data_matmul, weights_reshaped)
print(matmul_output.shape)

torch.Size([72, 224, 224])


In [8]:
# Define a simple CNN model with one convolutional layer
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1):
        super(SimpleCNN, self).__init__()
        
        # Convolutional layer
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,   # Number of input channels (e.g., 3 for RGB)
            out_channels=out_channels, # Number of output feature maps
            kernel_size=kernel_size,   # Size of the convolution kernel
            stride=stride,             # Step size of the convolution
            padding=padding,           # Padding to preserve spatial dimensions
            bias=False                 # disable biases
        )

    def forward(self, x):
        # Apply convolution
        x = self.conv1(x)
        return x


Method 2: Torch 2DConv

In [9]:

simple_cnn = SimpleCNN(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
image_data_cnn = image_data


def test_2dconv(cnn, image_data, weights):
    cnn.conv1.load_state_dict({'weight': weights}, strict=False)
    cnn.eval()
    with torch.no_grad():
        conv_output_torch =simple_cnn(image_data) 
    return conv_output_torch

cnn_output = test_2dconv(simple_cnn, image_data_cnn, weights)
print(cnn_output.shape)


torch.Size([72, 224, 224])


In [43]:
def test_comparison(cnn_output, matmul_output):

    # Print shapes for quick verification
    print(f"cnn_output shape: {cnn_output.shape}")
    print(f"matmul_output shape: {matmul_output.shape}")

    # Check if shapes match
    if cnn_output.shape != matmul_output.shape:
        print("Shapes do not match! Cannot compare.")
    else:
        # Check for exact equality (rarely true due to floating-point)
        exact_match = torch.equal(cnn_output, matmul_output)
        print(f"Exact equality: {exact_match}")
        
        # Check for approximate equality (recommended for floating-point tensors)
        close_match = torch.allclose(cnn_output, matmul_output, atol=1e-4, rtol=1e-3)
        print(f"Approximate equality (allclose): {close_match}")
        
        if not close_match and not exact_match:
            # Compute and print the maximum absolute difference
            diff = torch.abs(cnn_output - matmul_output)
            max_diff = torch.max(diff)
            print(f"Maximum absolute difference: {max_diff}")
            
            # Optionally, print mean difference
            mean_diff = torch.mean(diff)
            print(f"Mean absolute difference: {mean_diff}")
            
        
            numerator = torch.abs(cnn_output - matmul_output)
            denominator = (torch.abs(cnn_output) + torch.abs(matmul_output)) / 2
        
            # Avoid division by zero
            percent_diff = torch.where(denominator != 0, (numerator / denominator) * 100, torch.zeros_like(denominator))

            print(f"Maximum percentage difference: {torch.max(percent_diff)}%")
            return diff,False
        else:
            return None,True
        
def percentage_approx_equal(tensor_a: torch.Tensor, tensor_b: torch.Tensor, tol: float = 1e-3) -> float:
    """
    Calculate the percentage of elements in two tensors that are approximately equal.

    Args:
        tensor_a (torch.Tensor): First tensor.
        tensor_b (torch.Tensor): Second tensor.
        tol (float): Absolute tolerance for comparison.

    Returns:
        float: Percentage of approximately equal elements (0.0 to 100.0).
    """
    # Validate shapes
    if tensor_a.shape != tensor_b.shape:
        raise ValueError(f"Shape mismatch: {tensor_a.shape} vs {tensor_b.shape}")

    # Ensure both tensors are on the same device and dtype
    tensor_a = tensor_a.to(dtype=torch.float32)
    tensor_b = tensor_b.to(dtype=torch.float32)

    # Compare using torch.isclose (handles floating point tolerance)
    close_mask = torch.isclose(tensor_a, tensor_b, atol=tol, rtol=0.0)

    # Calculate percentage
    total_elements = close_mask.numel()
    matching_elements = close_mask.sum().item()
    percentage = (matching_elements / total_elements) * 100

    return percentage

def find_not_equal_indices(tensor_a: torch.Tensor, tensor_b: torch.Tensor, tol: float = 1e-5):
    """
    Find indices of elements in two 3D tensors that are NOT approximately equal.

    Args:
        tensor_a (torch.Tensor): First 3D tensor.
        tensor_b (torch.Tensor): Second 3D tensor.
        tol (float): Absolute tolerance for comparison.

    Returns:
        torch.Tensor: Indices of elements that are not approximately equal.
    """
    # Validate shapes
    if tensor_a.shape != tensor_b.shape:
        raise ValueError(f"Shape mismatch: {tensor_a.shape} vs {tensor_b.shape}")

    # Ensure float type for comparison
    tensor_a = tensor_a.to(dtype=torch.float32)
    tensor_b = tensor_b.to(dtype=torch.float32)

    # Boolean mask for elements that are NOT close
    not_close_mask = ~torch.isclose(tensor_a, tensor_b, atol=tol, rtol=0.0)

    # Get indices of mismatches
    mismatch_indices = torch.nonzero(not_close_mask, as_tuple=False)

    return mismatch_indices
test_comparison(cnn_output, matmul_output)

cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): True


(None, True)

In [13]:
# test random weight equivalence and image data equivalence for matmul and 2dconv

def test_1():
    # define random weights and image data for testing
    weights_random = torch.randn((out_channels, in_channels, kernel_size, kernel_size), dtype=data_type)
    weights_random_reshaped = weights_random.view(weights_random.size(0), -1)
    image_raw = get_image(image_path)

    # apply transformations to random data
    image_data = nn_transform(image_raw, size=(image_H, image_L), data_type=data_type)
    image_data_cnn = image_data

    image_data_matmul = matmul_transform(image_data, kernel_size=kernel_size, stride=stride, padding=padding, data_type=data_type)

    # test matmul and 2dconv with random data
    matmul_output_random = test_matmult(image_data_matmul, weights_random_reshaped)
    cnn_output_random = test_2dconv(simple_cnn, image_data_cnn, weights_random)

    #verify outputs are close
    equal = test_comparison(cnn_output_random, matmul_output_random)
    assert equal, "Test failed: Outputs are not approximately equal."

# run 10 tests to ensure equlity with random weights    
for i in range(10):
    test_1()
    print(f"Test {i+1} passed successfully.")


cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): True
Test 1 passed successfully.
cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): True
Test 2 passed successfully.
cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): True
Test 3 passed successfully.
cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): True
Test 4 passed successfully.
cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): True
Test 5 passed successfully.
cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Ex

Method 3: use systolic array

In [91]:
import numpy as np

class PE:
    def __init__(self):
        self.a = 0.0
        self.b = 0.0
        self.acc = 0.0

    def step(self, a_in, b_in):
        # MAC with previous cycle's a,b
        self.acc += self.a * self.b
        a_out = self.a
        b_out = self.b
        self.a = a_in
        self.b = b_in
        return a_out, b_out

class SystolicArray:
    def __init__(self, P, Q):
        self.P = P  # rows (output rows per tile)
        self.Q = Q  # cols (output cols per tile)
        self.pes = [[PE() for _ in range(Q)] for _ in range(P)]

    def run_tile(self, A_block, B_block):
        """
        A_block: (P x Kb)
        B_block: (Kb x Q)
        Returns: C_block: (P x Q)
        """
        P, Kb = A_block.shape
        Kb2, Q = B_block.shape
        assert P == self.P and Q == self.Q and Kb == Kb2

        T = Kb + P + Q - 1  # pipeline fill + drain

        A_stream = torch.zeros((T, P), dtype=A_block.dtype)
        B_stream = torch.zeros((T, Q), dtype=B_block.dtype)

        # Wavefront schedule
        for t in range(T):
            for i in range(P):
                k = t - i
                if 0 <= k < Kb:
                    A_stream[t, i] = A_block[i, k]
            for j in range(Q):
                k = t - j
                if 0 <= k < Kb:
                    B_stream[t, j] = B_block[k, j]

        for t in range(T):
            a_next = [[0.0]*self.Q for _ in range(self.P)]
            b_next = [[0.0]*self.Q for _ in range(self.P)]
            for i in range(self.P):
                for j in range(self.Q):
                    if j == 0:
                        a_in = A_stream[t, i]
                    else:
                        a_in = a_next[i][j-1]

                    if i == 0:
                        b_in = B_stream[t, j]
                    else:
                        b_in = b_next[i-1][j]

                    a_out, b_out = self.pes[i][j].step(a_in, b_in)
                    a_next[i][j] = a_out
                    b_next[i][j] = b_out

        C_block = torch.zeros((self.P, self.Q), dtype=A_block.dtype)
        for i in range(self.P):
            for j in range(self.Q):
                C_block[i, j] = self.pes[i][j].acc
                self.pes[i][j].acc = 0.0
                self.pes[i][j].a = 0.0
                self.pes[i][j].b = 0.0
        return C_block
    
def im2col(image, kernel_size=3, stride=1, padding=1):
    """
    image: (C, H, W)
    Returns: (C * Kh * Kw, H_out * W_out)
    """
    C, H, W = image.shape
    Kh = Kw = kernel_size

    H_out = (H + 2*padding - Kh) // stride + 1
    W_out = (W + 2*padding - Kw) // stride + 1

    padded = np.zeros((C, H + 2*padding, W + 2*padding), dtype=image.dtype)
    padded[:, padding:padding+H, padding:padding+W] = image

    cols = np.zeros((C * Kh * Kw, H_out * W_out), dtype=image.dtype)

    col_idx = 0
    for y in range(0, H_out):
        for x in range(0, W_out):
            patch = padded[:, y*stride:y*stride+Kh, x*stride:x*stride+Kw]
            cols[:, col_idx] = patch.reshape(-1)
            col_idx += 1
    return cols  # K x N


def systolic_gemm(A, B, P=16, Q=16, Kb=16):
    """
    A: (M x K), B: (K x N) -> C: (M x N)
    Tiled over a P x Q systolic array, inner K tile size Kb.
    Assumes divisibility for simplicity.
    """
    M, K = A.shape
    K2, N = B.shape
    assert K == K2, f'Inner dimensions must match: {K} vs {K2}'
    assert M % P == 0, f'{M} not divisible by {P}'
    assert N % Q == 0, f'{N} not divisible by {Q}'
    assert K % Kb == 0, f'{K} not divisible by {Kb}'

    sa = SystolicArray(P, Q)
    C = torch.zeros((M, N), dtype=A.dtype)

    for i0 in range(0, M, P):
        for j0 in range(0, N, Q):
            C_block = torch.zeros((P, Q), dtype=A.dtype)
            for k0 in range(0, K, Kb):
                A_block = A[i0:i0+P, k0:k0+Kb]
                B_block = B[k0:k0+Kb, j0:j0+Q]
                C_block += sa.run_tile(A_block, B_block)
            C[i0:i0+P, j0:j0+Q] = C_block
            print(f"Completed row block row: {i0}/{M}, col: {j0}/{N}")
    return C


In [49]:
def test_systolic(weights, image_data, P, Q, Kb):
    systolic_output = systolic_gemm(weights, image_data, P=P, Q=Q, Kb=Kb)
    # Compute H_out and W_out correctly
    H_out = ((image_H + 2 * padding - kernel_size) // stride) + 1
    W_out = ((image_L + 2 * padding - kernel_size) // stride) + 1

    # Reshape matmul_output directly (no need for F.fold)
    systolic_output = systolic_output.view(out_channels, H_out, W_out)
    return systolic_output
systolic_output = test_systolic( weights_reshaped,image_data_matmul, P=36, Q=28, Kb=27)

Completed row block row: 0/72, col: 0/50176
Completed row block row: 0/72, col: 16/50176
Completed row block row: 0/72, col: 32/50176
Completed row block row: 0/72, col: 48/50176
Completed row block row: 0/72, col: 64/50176
Completed row block row: 0/72, col: 80/50176
Completed row block row: 0/72, col: 96/50176
Completed row block row: 0/72, col: 112/50176
Completed row block row: 0/72, col: 128/50176
Completed row block row: 0/72, col: 144/50176
Completed row block row: 0/72, col: 160/50176
Completed row block row: 0/72, col: 176/50176
Completed row block row: 0/72, col: 192/50176
Completed row block row: 0/72, col: 208/50176
Completed row block row: 0/72, col: 224/50176
Completed row block row: 0/72, col: 240/50176
Completed row block row: 0/72, col: 256/50176
Completed row block row: 0/72, col: 272/50176
Completed row block row: 0/72, col: 288/50176
Completed row block row: 0/72, col: 304/50176
Completed row block row: 0/72, col: 320/50176
Completed row block row: 0/72, col: 336/50

In [45]:
print("Comparing CNN output and Systolic Array output...")
diff_cnn,systolic_match = test_comparison(cnn_output, systolic_output)

print(percentage_approx_equal(cnn_output, systolic_output))

indexes = find_not_equal_indices(cnn_output, systolic_output, tol=1e-3).tolist()
print(indexes)
list_cnn = cnn_output.tolist()
list_systolic = systolic_output.tolist()

max_values, max_indices = torch.max(diff_cnn, dim=0)
print(torch.max(max_indices))
print("Comparing Systolic Array output and MatMul output...")
diff_systolic,matmul_match = test_comparison(systolic_output, matmul_output)

print("Comparing CNN output and MatMul output...")
diff_cnm,matmul_match = test_comparison(cnn_output, matmul_output)

Comparing CNN output and Systolic Array output...
cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): False
Maximum absolute difference: 2.4134178161621094
Mean absolute difference: 0.0010147782741114497
Maximum percentage difference: 200.0%
99.91358196924604
tensor(35)
Comparing Systolic Array output and MatMul output...
cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): False
Maximum absolute difference: 2.413421630859375
Mean absolute difference: 0.001012488268315792
Maximum percentage difference: 200.0%
Comparing CNN output and MatMul output...
cnn_output shape: torch.Size([72, 224, 224])
matmul_output shape: torch.Size([72, 224, 224])
Exact equality: False
Approximate equality (allclose): True


Test 2: Manually verify small array against FPGA systolic array

In [84]:
import torch

# Constants
kernel_size = 3
stride = 1
padding = 1

image_H = 3
image_L = 3
image_path = r"C:\Users\dwmcc\OneDrive\Documents\GitHub\SystolicArrayTransform\images\000320150002.jpg"
data_type = torch.float32  # Change to torch.int8 for int8 data type

in_channels = 1
out_channels = 1

In [93]:
image_data = [[[1, 2, 3],
               [4, 5, 6],
                [7, 8, 9]]]
weights = [[[[9,8,7],
            [6,5,4], 
            [3,2,1]]]]
cnn_image_data = torch.tensor(image_data, dtype=torch.float32)  # (1, H, W)
matmul_image_data = F.unfold(cnn_image_data, kernel_size=kernel_size, stride=stride, padding=padding)  # (C*Kh*Kw, H_out*W_out)
matmul_image_data = matmul_image_data.to(data_type)

weights_tensor = torch.tensor(weights, dtype=torch.float32)  # (out_channels, in_channels, kernel_size, kernel_size)
weights_tensor_reshaped = weights_tensor.view(out_channels, -1).to(data_type)  # (out_channels, in_channels*kernel_size*kernel_size)
simple_cnn = SimpleCNN(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
output_cnn = test_2dconv(simple_cnn, cnn_image_data, weights_tensor).tolist()
output_matmul = test_matmult(matmul_image_data, weights_tensor_reshaped).tolist()
output_systolic = test_systolic( weights_tensor_reshaped,matmul_image_data, P=1, Q=1, Kb=1).tolist()

print("CNN Output:", output_cnn)
print("MatMul Output:", output_matmul) 
print("Systolic Output:", output_systolic)



Completed row block row: 0/1, col: 0/9
Completed row block row: 0/1, col: 1/9
Completed row block row: 0/1, col: 2/9
Completed row block row: 0/1, col: 3/9
Completed row block row: 0/1, col: 4/9
Completed row block row: 0/1, col: 5/9
Completed row block row: 0/1, col: 6/9
Completed row block row: 0/1, col: 7/9
Completed row block row: 0/1, col: 8/9
CNN Output: [[[26.0, 56.0, 54.0], [84.0, 165.0, 144.0], [134.0, 236.0, 186.0]]]
MatMul Output: [[[26.0, 56.0, 54.0], [84.0, 165.0, 144.0], [134.0, 236.0, 186.0]]]
Systolic Output: [[[26.0, 56.0, 54.0], [84.0, 165.0, 144.0], [134.0, 236.0, 186.0]]]
